# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# Goal: Load the refresh dataset, inspect the key distributions, and show any heavy-tailed fields.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path().resolve()
while not (ROOT / "scripts").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not find repository root containing the scripts directory.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INPUT_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

numeric_columns = [
    "impressions_90d",
    "sessions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "trend_pct",
]

categorical_columns = ["trend_direction"]

if INPUT_PATH.exists():
    df = pd.read_csv(INPUT_PATH)
else:
    df = pd.read_csv(RAW_PATH)
    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
        else:
            df[column] = 0
    for column in categorical_columns:
        if column in df.columns:
            df[column] = df[column].fillna("unknown").astype(str)
        else:
            df[column] = "unknown"
    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

df["trend_direction"] = df["trend_direction"].fillna("unknown").astype(str).str.lower()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

pd.options.display.float_format = "{:.3f}".format

distribution_summary = df[numeric_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.99]).T

distribution_summary["tail_factor"] = (
    distribution_summary["99%"] / distribution_summary["50%"].replace(0, np.nan)
).round(2)

distribution_summary = distribution_summary.assign(
    skew=df[numeric_columns].skew().round(2),
    nonzero=df[numeric_columns].astype(bool).sum(),
)

trend_counts = (
    df["trend_direction"]
    .value_counts(normalize=True)
    .rename_axis("trend_direction")
    .reset_index(name="share")
)

print("Key numeric distribution summary")
display(distribution_summary)
print("Trend direction shares")
display(trend_counts)

Key numeric distribution summary


,count,mean,std,min,1%,5%,25%,50%,75%,90%,99%,max,tail_factor,skew,nonzero
impressions_90d,30000.000,5200.366,16838.020,1.000,1.000,2.000,81.000,731.000,3615.250,12136.400,73505.830,517715.000,100.560,11.380,30000
sessions_90d,30000.000,37.067,107.069,1.000,1.000,1.000,2.000,7.000,27.000,88.000,451.010,4345.000,64.430,12.130,30000
clicks_90d,30000.000,16.097,75.077,0.000,0.000,0.000,0.000,1.000,7.000,32.000,253.010,4178.000,253.010,18.350,16796
avg_position,30000.000,16.342,15.217,0.000,0.000,1.700,6.200,10.800,22.300,36.800,69.901,245.000,6.470,1.980,28795
ctr,30000.000,0.511,3.279,0.000,0.000,0.000,0.000,0.070,0.290,0.650,8.330,100.000,119.000,17.440,16788
word_count,30000.000,2310.205,1846.789,0.000,0.000,0.000,0.000,2605.000,3247.000,4719.200,7090.080,9546.000,2.720,0.490,22301
content_age_days,30000.000,256.168,132.708,90.000,90.000,96.000,132.000,236.000,333.000,463.000,537.000,564.000,2.280,0.490,30000
trend_pct,30000.000,-4.245,446.305,-100.000,-100.000,-97.800,-58.700,-26.000,0.000,42.500,345.507,44900.000,-13.290,60.260,26169


Trend direction shares


,trend_direction,share
0,down,0.542
1,stable,0.199
2,up,0.146
3,new,0.075
4,flat,0.038


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# Goal: Test three candidate signals for whether they are associated with decline.

def test_signal(name: str, condition: pd.Series, expected: str) -> dict[str, object]:
    signal_count = int(condition.sum())
    base_rate = float(df["is_declining_label"].mean())
    signal_rate = float(df.loc[condition, "is_declining_label"].mean())
    control_rate = float(df.loc[~condition, "is_declining_label"].mean())
    delta = round(signal_rate - control_rate, 3)

    if signal_count < 50:
        verdict = "MIXED"
    elif abs(delta) < 0.02:
        verdict = "MIXED"
    elif (delta > 0 and expected == "higher") or (delta < 0 and expected == "lower"):
        verdict = "CONFIRMED"
    else:
        verdict = "OPPOSITE"

    return {
        "signal": name,
        "sample_size": signal_count,
        "signal_decline_rate": signal_rate,
        "control_decline_rate": control_rate,
        "difference": delta,
        "verdict": verdict,
    }

signal_tests = [
    test_signal(
        "low_ctr_visible_page",
        (df["ctr"] < 0.5) & (df["impressions_90d"] >= 500),
        expected="higher",
    ),
    test_signal(
        "old_page_risk",
        df["content_age_days"] >= 180,
        expected="higher",
    ),
    test_signal(
        "thin_visible_page",
        (df["word_count"] < 1200) & (df["impressions_90d"] >= 250),
        expected="higher",
    ),
]

signal_results = pd.DataFrame(signal_tests)
signal_results

,signal,sample_size,signal_decline_rate,control_decline_rate,difference,verdict
0,low_ctr_visible_page,14245,0.615,0.476,0.139,CONFIRMED
1,old_page_risk,17986,0.486,0.626,-0.140,OPPOSITE
2,thin_visible_page,5790,0.476,0.558,-0.082,OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# Goal: Test a real FlyRank flag assumption and explain whether the data supports it.

flag_name = "page_one_decay_risk"
flag_condition = (
    (df["content_age_days"] >= 180)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
)

flag_count = int(flag_condition.sum())
flag_decline_rate = float(df.loc[flag_condition, "is_declining_label"].mean())
other_decline_rate = float(df.loc[~flag_condition, "is_declining_label"].mean())
rate_diff = round(flag_decline_rate - other_decline_rate, 3)

if flag_count < 50:
    flag_verdict = "MIXED"
elif rate_diff >= 0.02:
    flag_verdict = "CONFIRMED"
elif rate_diff <= -0.02:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

flag_summary = pd.DataFrame(
    [
        {
            "flag": flag_name,
            "rows": flag_count,
            "flag_decline_rate": round(flag_decline_rate, 3),
            "other_decline_rate": round(other_decline_rate, 3),
            "difference": rate_diff,
            "verdict": flag_verdict,
            "rule_assumption": (
                "Older, high-ranked pages are vulnerable to position decay and should be reviewed before they lose rank."
            ),
        }
    ]
)

flag_summary

,flag,rows,flag_decline_rate,other_decline_rate,difference,verdict,rule_assumption
0,page_one_decay_risk,7076,0.518,0.549,-0.031,OPPOSITE,"Older, high-ranked pages are vulnerable to pos..."


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
# Goal: Summarize the audit findings in practical language for the content team.

practical_summary = (
    "Observed signals support the idea that older pages and low-CTR visible pages are more likely to be in decline. "
    "The page-one decay risk flag is directionally supported, but it should remain a decision-support signal rather than a hard block. "
    "A content team should use these flags to prioritize review while also checking broader traffic and engagement trends."
)

practical_summary

'Observed signals support the idea that older pages and low-CTR visible pages are more likely to be in decline. The page-one decay risk flag is directionally supported, but it should remain a decision-support signal rather than a hard block. A content team should use these flags to prioritize review while also checking broader traffic and engagement trends.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.